In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
from torchvision.datasets import MNIST
from torch.utils.data import DataLoader
from torchvision.transforms import transforms

In [2]:
transform = transforms.Compose([
    transforms.ToTensor()
])

trainset = MNIST(root = "./data",train =True ,download = True,transform = transform)
testset = MNIST(root="./data",train = False ,download =True , transform = transform)

In [3]:
trainloader = DataLoader(trainset,shuffle = True , batch_size = 64)
testloader = DataLoader(testset,shuffle = True,batch_size = 64)

# Building CNN

In [4]:
class CNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.conv_layers = nn.Sequential(
            nn.Conv2d(1,32,kernel_size=3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2,2),

            nn.Conv2d(32,64,kernel_size=3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2,2),

            nn.Conv2d(64,128,kernel_size=3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2,2),

        )


        self.fc_layers = nn.Sequential(
            nn.Linear(3*3*128,256),nn.ReLU(),

            nn.Linear(256,10)
        )

    def forward(self,x):
        x = self.conv_layers(x)
        x = x.view(x.size(0),-1) #flattening
        x = self.fc_layers(x)

        return x

In [5]:
model = CNN()
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

# Training and evaluation

In [6]:
epochs = 10
for epoch in range(epochs):
    epoch_training_loss = 0.0

    for images,labels in trainloader:
        optimizer.zero_grad()
        output = model.forward(images)
        loss = criterion(output,labels)
        loss.backward()

        optimizer.step() #update parameter
        epoch_training_loss += loss.item()

    model.eval()
    running_validation_loss = 0.0
    
    with torch.no_grad():
        for images,labels in testloader:
            output = model(images)
            loss = criterion(output,labels)
            running_validation_loss += loss.item()

    epoch_val_loss = running_validation_loss/ len(testloader)
    print(f"epoch{epoch+1}{epochs} and loss ={epoch_training_loss/len(trainloader)} and validation loss = {epoch_val_loss}")
    

epoch110 and loss =0.1694972178420183 and validation loss = 0.042374206431785445
epoch210 and loss =0.04536037201405382 and validation loss = 0.03303239073576168
epoch310 and loss =0.032028868553683046 and validation loss = 0.046230936600162295
epoch410 and loss =0.023396758035038857 and validation loss = 0.03846246800214871
epoch510 and loss =0.01865562495130125 and validation loss = 0.036010872531878006


KeyboardInterrupt: 

In [12]:
correct_labels = 0
total_labels = 0

with torch.no_grad():
    for images,labels in testloader:
        outputs = model.forward(images)
        _,predicted = torch.max(outputs,1)

        correct_labels += (predicted == labels).sum().item()
        total_labels += labels.size(0)
print(f"accuracy{correct_labels/total_labels *100}")

accuracy99.24


# Building RNN 


In [16]:
class RNN(nn.Module):
    def __init__(self,input_size=28,hidden_size=128,num_layers=1):
        super().__init__()

        self.hidden_size = hidden_size
        self.num_layers = num_layers

        self.rnn = nn.RNN(input_size = input_size,hidden_size = hidden_size,num_layers=num_layers,batch_first=True)
        #fclayers
        self.fc = nn.Linear(hidden_size,10)


    def forward(self,x):
        out,_ = self.rnn(x)
        out = self.fc(out[:,-1,:])
        return out



        

In [18]:
model = RNN()
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

In [7]:
epochs = 10
for epoch in range(epochs):
    model.train()
    epoch_training_loss = 0.0

    for images,labels in trainloader:
        optimizer.zero_grad()
        images = images.squeeze(1)
    
        outputs = model(images)
        loss = criterion(outputs,labels)
        loss.backward()
        optimizer.step() #update parameter
        epoch_training_loss += loss.item()

    model.eval()
    running_validation_loss = 0.0
    
    with torch.no_grad():
        for images,labels in testloader:
            images = images.squeeze(1)
            output = model(images)
            loss = criterion(output,labels)
            running_validation_loss += loss.item()

    epoch_train_loss = epoch_training_loss / len(trainloader)
    epoch_val_loss = running_validation_loss/ len(testloader)
   
    
    print(
        f"Epoch {epoch+1}/{epochs} | "
        f"Train Loss: {epoch_train_loss:.4f} | "
        f"Validation Loss: {epoch_val_loss:.4f}"
    )

RuntimeError: Given groups=1, weight of size [32, 1, 3, 3], expected input[1, 64, 28, 28] to have 1 channels, but got 64 channels instead

In [24]:
with torch.no_grad():
    correct_vals = 0
    total_vals = 0 

    for images,labels in testloader:
        images = images.squeeze(1)

        outputs = model(images)
        _,predicted = torch.max(outputs,1)

        total_vals += labels.size(0)
        correct_vals += (predicted == labels).sum().item()
    accuracy = 100 * correct_vals/total_vals
    print(f"Accuracy= {accuracy:.2f}%")

Accuracy= 96.88%
